# Week 2 Exercise Solution - Technical Q/A ChatBot

Building a learning chatbot. Users can switch from one teacher to another.

- Physics teacher
- Chemistry teacher
- Math teacher

Each teacher has a different teaching style and expertise.
Whenever a user switches teacher, the conversation context should be reset.

The chatbot should be able to use tools to help the user learn.

In [ ]:
import os
import math
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


In [ ]:
# Initialization and API key setup
load_dotenv(override=True)

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}")
else: 
  print("OpenRouter API Key not set")

In [ ]:
# OpenAI Client
def client():
  return OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")

In [ ]:
# Models
models = ["gpt-4.1-mini", "anthropic/claude-haiku-4.5"]

In [ ]:
# Current Teacher state
current_teacher = "physics"
# Teachers
teachers = ["physics", "chemistry", "math"]

In [ ]:
# Prompts
system_prompt = """
You are a high school teacher helping a student with questions in your subject.

Core rules:
- Only answer questions related to your subject. If asked about another subject or off-topic, politely redirect or say you focus on your subject.
- Be supportive and encourage curiosity.
- Keep answers appropriate for high school level—clear, accurate, and not overly technical unless the student asks for depth.
- When tools are available, use them when they will help (calculations, lookups, conversions). Don't force tool use when a direct answer is enough.

Teacher personality and style:
{teacher_prompt}
"""

# Teacher prompts
physics_teacher_prompt = """
Subject: Physics

Personality: Enthusiastic and curious. Treats physics as a way to understand how the world works. Loves "aha" moments and connecting theory to everyday life.

Teaching style:
- Use everyday analogies (e.g., roller coasters for energy, sports for motion).
- Explain step by step, then connect to the bigger picture.
- Reference experiments, thought experiments, and real-world applications.
- Encourage questions and "what if" thinking.

Tone: Friendly, energetic, slightly nerdy. Phrases like "Here’s the cool part…" or "Think of it this way…"
"""

chemistry_teacher_prompt = """
Subject: Chemistry

Personality: Methodical and precise. Thinks in structures, equations, and processes. Values accuracy and clarity.

Teaching style:
- Get to the point quickly; avoid unnecessary tangents.
- Emphasize patterns (periodic table, reaction types, bonding rules).
- Use clear formulas and step-by-step logic.
- When relevant, mention lab safety and practical applications.

Tone: Calm, confident. Phrases like "The key point is…" or "In chemical terms…"
"""

math_teacher_prompt = """
Subject: Mathematics

Personality: Logical and methodical. Cares about clear steps, structure, and reasoning. Explains why, not just how.

Teaching style:
- Always use the available tools for calculations (area, etc.)—do not compute by hand.
- After getting the tool result, explain the steps and reasoning clearly.
- Show the formula and substitution, then present the tool's result.
- Use scientific notation for very large (≥10,000) or very small (<0.01) values.

Tone: Patient, precise. Phrases like "Let’s work through this…" or "Notice that…"
"""

# Teacher prompts map
teacher_prompt_map = {
  "physics": physics_teacher_prompt,
  "chemistry": chemistry_teacher_prompt,
  "math": math_teacher_prompt,
}

### Tools Functions and their descriptions

In [ ]:
# Periodic Table
periodic_table = {
    "Hydrogen":   {"symbol": "H", "number": 1},
    "Helium":     {"symbol": "He", "number": 2},
    "Lithium":    {"symbol": "Li", "number": 3},
    "Beryllium":  {"symbol": "Be", "number": 4},
    "Boron":      {"symbol": "B", "number": 5},
    "Carbon":     {"symbol": "C", "number": 6},
    "Nitrogen":   {"symbol": "N", "number": 7},
    "Oxygen":     {"symbol": "O", "number": 8},
    "Fluorine":   {"symbol": "F", "number": 9},
    "Neon":       {"symbol": "Ne", "number": 10},
    "Sodium":     {"symbol": "Na", "number": 11},
    "Magnesium":  {"symbol": "Mg", "number": 12},
    "Aluminum":   {"symbol": "Al", "number": 13},
    "Silicon":    {"symbol": "Si", "number": 14},
    "Phosphorus": {"symbol": "P", "number": 15},
    "Sulfur":     {"symbol": "S", "number": 16},
    "Chlorine":   {"symbol": "Cl", "number": 17},
    "Argon":      {"symbol": "Ar", "number": 18},
    "Potassium":  {"symbol": "K", "number": 19},
    "Calcium":    {"symbol": "Ca", "number": 20},
    "Scandium":   {"symbol": "Sc", "number": 21},
    "Titanium":   {"symbol": "Ti", "number": 22},
    "Vanadium":   {"symbol": "V", "number": 23},
    "Chromium":   {"symbol": "Cr", "number": 24},
    "Manganese":  {"symbol": "Mn", "number": 25},
    "Iron":       {"symbol": "Fe", "number": 26},
    "Cobalt":     {"symbol": "Co", "number": 27},
    "Nickel":     {"symbol": "Ni", "number": 28},
    "Copper":     {"symbol": "Cu", "number": 29},
    "Zinc":       {"symbol": "Zn", "number": 30},
    "Gallium":    {"symbol": "Ga", "number": 31},
    "Germanium":  {"symbol": "Ge", "number": 32},
    "Arsenic":    {"symbol": "As", "number": 33},
    "Selenium":   {"symbol": "Se", "number": 34},
    "Bromine":    {"symbol": "Br", "number": 35},
    "Krypton":    {"symbol": "Kr", "number": 36},
    "Rubidium":   {"symbol": "Rb", "number": 37},
    "Strontium":  {"symbol": "Sr", "number": 38},
    "Yttrium":    {"symbol": "Y", "number": 39},
    "Zirconium":  {"symbol": "Zr", "number": 40},
    "Niobium":    {"symbol": "Nb", "number": 41},
    "Molybdenum": {"symbol": "Mo", "number": 42},
    "Technetium": {"symbol": "Tc", "number": 43},
    "Ruthenium":  {"symbol": "Ru", "number": 44},
    "Rhodium":    {"symbol": "Rh", "number": 45},
    "Palladium":  {"symbol": "Pd", "number": 46},
    "Silver":     {"symbol": "Ag", "number": 47},
    "Cadmium":    {"symbol": "Cd", "number": 48},
    "Indium":     {"symbol": "In", "number": 49},
    "Tin":        {"symbol": "Sn", "number": 50},
    "Antimony":   {"symbol": "Sb", "number": 51},
    "Tellurium":  {"symbol": "Te", "number": 52},
    "Iodine":     {"symbol": "I", "number": 53},
    "Xenon":      {"symbol": "Xe", "number": 54},
    "Cesium":     {"symbol": "Cs", "number": 55},
    "Barium":     {"symbol": "Ba", "number": 56},
    "Lanthanum":  {"symbol": "La", "number": 57},
    "Cerium":     {"symbol": "Ce", "number": 58},
    "Praseodymium": {"symbol": "Pr", "number": 59},
    "Neodymium":  {"symbol": "Nd", "number": 60},
    "Promethium": {"symbol": "Pm", "number": 61},
    "Samarium":   {"symbol": "Sm", "number": 62},
    "Europium":   {"symbol": "Eu", "number": 63},
    "Gadolinium": {"symbol": "Gd", "number": 64},
    "Terbium":    {"symbol": "Tb", "number": 65},
    "Dysprosium": {"symbol": "Dy", "number": 66},
    "Holmium":    {"symbol": "Ho", "number": 67},
    "Erbium":     {"symbol": "Er", "number": 68},
    "Thulium":    {"symbol": "Tm", "number": 69},
    "Ytterbium":  {"symbol": "Yb", "number": 70},
    "Lutetium":   {"symbol": "Lu", "number": 71},
    "Hafnium":    {"symbol": "Hf", "number": 72},
    "Tantalum":   {"symbol": "Ta", "number": 73},
    "Tungsten":   {"symbol": "W", "number": 74},
    "Rhenium":    {"symbol": "Re", "number": 75},
    "Osmium":     {"symbol": "Os", "number": 76},
    "Iridium":    {"symbol": "Ir", "number": 77},
    "Platinum":   {"symbol": "Pt", "number": 78},
    "Gold":       {"symbol": "Au", "number": 79},
    "Mercury":    {"symbol": "Hg", "number": 80},
    "Thallium":   {"symbol": "Tl", "number": 81},
    "Lead":       {"symbol": "Pb", "number": 82},
    "Bismuth":    {"symbol": "Bi", "number": 83},
    "Polonium":   {"symbol": "Po", "number": 84},
    "Astatine":   {"symbol": "At", "number": 85},
    "Radon":      {"symbol": "Rn", "number": 86},
    "Francium":   {"symbol": "Fr", "number": 87},
    "Radium":     {"symbol": "Ra", "number": 88},
    "Actinium":   {"symbol": "Ac", "number": 89},
    "Thorium":    {"symbol": "Th", "number": 90},
    "Protactinium": {"symbol": "Pa", "number": 91},
    "Uranium":    {"symbol": "U", "number": 92},
    "Neptunium":  {"symbol": "Np", "number": 93},
    "Plutonium":  {"symbol": "Pu", "number": 94},
    "Americium":  {"symbol": "Am", "number": 95},
    "Curium":     {"symbol": "Cm", "number": 96},
    "Berkelium":  {"symbol": "Bk", "number": 97},
    "Californium": {"symbol": "Cf", "number": 98},
    "Einsteinium": {"symbol": "Es", "number": 99},
    "Fermium":    {"symbol": "Fm", "number": 100},
    "Mendelevium": {"symbol": "Md", "number": 101},
    "Nobelium":   {"symbol": "No", "number": 102},
    "Lawrencium": {"symbol": "Lr", "number": 103},
    "Rutherfordium": {"symbol": "Rf", "number": 104},
    "Dubnium":    {"symbol": "Db", "number": 105},
    "Seaborgium": {"symbol": "Sg", "number": 106},
    "Bohrium":    {"symbol": "Bh", "number": 107},
    "Hassium":    {"symbol": "Hs", "number": 108},
    "Meitnerium": {"symbol": "Mt", "number": 109},
    "Darmstadtium": {"symbol": "Ds", "number": 110},
    "Roentgenium": {"symbol": "Rg", "number": 111},
    "Copernicium": {"symbol": "Cn", "number": 112},
    "Nihonium":   {"symbol": "Nh", "number": 113},
    "Flerovium":  {"symbol": "Fl", "number": 114},
    "Moscovium":  {"symbol": "Mc", "number": 115},
    "Livermorium": {"symbol": "Lv", "number": 116},
    "Tennessine": {"symbol": "Ts", "number": 117},
    "Oganesson":  {"symbol": "Og", "number": 118},
}

In [ ]:
# Calculate Kinetic Energy
def calculate_kinetic_energy(mass, velocity):
    kinetic_energy = 0.5 * mass * velocity ** 2
    # Always include scientific notation for clarity
    sci_notation = f"{kinetic_energy:.2e}".replace("e+0", "×10^").replace("e-0", "×10^-").replace("e+", "×10^").replace("e-", "×10^-")
    return (
        f"Result: {kinetic_energy:.2f} J ({sci_notation})\n"
        f"Formula: KE = 0.5 × {mass} kg × ({velocity} m/s)²"
    )
    
# Build a symbol -> name mapping for lookup by symbol
SYMBOL_TO_NAME = {v["symbol"]: k for k, v in periodic_table.items()}

def periodic_table_lookup(element):
    """Look up by element name (e.g., Hydrogen) or symbol (e.g., H)."""
    elem_clean = element.strip()
    # Try name first, then symbol (case-insensitive for symbol)
    info = periodic_table.get(elem_clean) or periodic_table.get(
        SYMBOL_TO_NAME.get(elem_clean.capitalize()) or 
        SYMBOL_TO_NAME.get(elem_clean.title())
    )
    if not info:
        # Try case-insensitive symbol match (He, he, HE)
        for sym, name in SYMBOL_TO_NAME.items():
            if sym.lower() == elem_clean.lower():
                info = periodic_table[name]
                elem_clean = name  # Use canonical name in output
                break
    if info:
        return (
            f"Element: {elem_clean} ({info['symbol']})\n"
            f"Atomic number: {info['number']}\n"
            f"Use the name '{elem_clean}' and symbol '{info['symbol']}' when presenting to the student."
        )
    return f"Element '{element}' not found. Use the exact element name or symbol from the periodic table."

# Calculate the area of a circle
def calculate_area_of_circle(radius, units="m"):
    """
    radius: in the given units (m, mm, cm)
    units: "m", "mm", or "cm"
    """
    area = math.pi * radius ** 2  # Area in (units)²
    
    # Scientific notation for very large or small values
    if abs(area) >= 1e4 or (0 < abs(area) < 1e-2 and area != 0):
        from math import log10, floor
        exp = floor(log10(abs(area)))
        coef = area / (10 ** exp)
        formatted = f"{coef:.2f} × 10^{exp}"
    else:
        formatted = f"{area:.2f}"
    
    unit_sq = f"{units}²" if units in ("m", "mm", "cm") else "m²"
    return (
        f"Result: {formatted} {unit_sq}\n"
        f"Formula: A = πr² = π × ({radius})² = {area:.6g} {unit_sq}"
    )

In [ ]:
kinetic_energy_function = {
    "name": "calculate_kinetic_energy",
    "description": (
        "Calculate the kinetic energy (KE) of an object using KE = 0.5 x mass x velocity². "
        "Use this tool for any kinetic energy calculation. Present the result in joules (J). "
        "For very large or very small values, use scientific notation (e.g., 2.25 x 10² J or 1.5 x 10⁻³ J)."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "mass": {
                "type": "number",
                "description": "Mass of the object in kilograms (kg)"
            },
            "velocity": {
                "type": "number",
                "description": "Velocity (speed) of the object in meters per second (m/s)"
            }
        },
        "required": ["mass", "velocity"],
        "additionalProperties": False
    }
}

periodic_table_function = {
    "name": "periodic_table_lookup",
    "description": (
        "Look up element data from the periodic table. Accepts either the element name "
        "(e.g., Hydrogen, Carbon) or chemical symbol (e.g., H, C, Fe). "
        "Returns atomic number, symbol, and name. When presenting to the student, "
        "always use the exact name and symbol returned by this tool."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "element": {
                "type": "string",
                "description": "Element name (e.g., Hydrogen) or chemical symbol (e.g., H, Fe)"
            }
        },
        "required": ["element"],
        "additionalProperties": False
    }
}

area_of_circle_function = {
    "name": "calculate_area_of_circle",
    "description": (
        "Calculate the area of a circle using A = πr². Use this tool whenever the user asks "
        "for circle area or radius-to-area. The radius and result use the same units (m, mm, cm). "
        "Use scientific notation for very large or small areas (e.g., 3.14 × 10² mm²)."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "radius": {
                "type": "number",
                "description": "Radius of the circle (use the same unit as in the question)"
            },
            "units": {
                "type": "string",
                "description": "Unit of the radius: 'm' (meters), 'mm' (millimeters), or 'cm' (centimeters)",
                "default": "m"
            }
        },
        "required": ["radius"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [
  {"type": "function", "function": kinetic_energy_function}, 
  {"type": "function", "function": periodic_table_function}, 
  {"type": "function", "function": area_of_circle_function}
]

tool_function_map = {
  "calculate_kinetic_energy": calculate_kinetic_energy,
  "periodic_table_lookup": periodic_table_lookup,
  "calculate_area_of_circle": calculate_area_of_circle
}

### Use Tools and Call Models

In [ ]:
def handle_tools_calls(message):
    responses = []
    for tool_call in message.tool_calls:
      # Get the function name and parameters
      func_name = tool_call.function.name
      params = json.loads(tool_call.function.arguments)
      if func_name in tool_function_map:
          func = tool_function_map[func_name]
          result = func(**params)
          responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
          })
    return responses

In [ ]:
def chat(model, teacher, history):
  history = [{"role":h["role"], "content":h["content"]} for h in history]
  messages = [{"role": "system", "content": system_prompt.format(teacher_prompt=teacher_prompt_map[teacher])}]
  messages.extend(history)
  response = client().chat.completions.create(model=model, messages=messages, tools=tools)
  
  while response.choices[0].finish_reason == "tool_calls":
    message = response.choices[0].message
    responses = handle_tools_calls(message)
    messages.append(message)
    messages.extend(responses)
    response = client().chat.completions.create(model=model, messages=messages, tools=tools)
    
  reply = response.choices[0].message.content
  history += [{"role":"assistant", "content":reply}]

  return history

### UI with Gradio

In [ ]:
def put_message_in_chatbot(message, history):
  return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
  gr.Markdown("## Learning Chatbot")
  gr.Markdown("### Select a teacher and ask a question")
  teacher_selector = gr.Dropdown(
    choices=teachers, 
    value=current_teacher, 
    label="Select a teacher"
  )
  chatbot = gr.Chatbot(height=500, type="messages")
  model_selector = gr.Dropdown(
    choices=models,
    value=models[0],
    label="Select a model"
  )
  message = gr.Textbox(label="Ask a question")
  
  # Clear chat when teacher changes
  def clear_chat_and_notify(teacher):
    """Clear chat and optionally show a welcome message for the new teacher."""
    return [{"role": "assistant", "content": f"Switched to {teacher} teacher. How can I help you today?"}]
  teacher_selector.change(fn=clear_chat_and_notify, inputs=[teacher_selector], outputs=[chatbot])
  
  # Chat flow
  message.submit(
    put_message_in_chatbot, 
    inputs=[message, chatbot], 
    outputs=[message, chatbot]
  ).then(
    chat, inputs=[model_selector, teacher_selector, chatbot], outputs=[chatbot]
  )
  
ui.launch(inbrowser=True)
  